In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import glob

In [ ]:
# data_folder_path = "/home/hsph/Downloads/DataStore/SS/ASIANPAINT/2024/"
# csv_path_list = glob.glob(data_folder_path + "*.csv")

In [9]:
def process_csv(csv_path): 
# Load the CSV file
    df = pd.read_csv(csv_path)

    # Convert 'Date Time' column to datetime if it's not already
    df['Date Time'] = pd.to_datetime(df['Date Time'])

    # Extract the date of the file (assumes all rows in the file are from the same day)
    file_date = df['Date Time'].dt.date.iloc[0]

    # Get the opening spot price (first row value of 'Spot')
    opening_spot = df.loc[0, 'Spot']

    # Calculate moneyness
    df['moneyness'] = df['Strike'] / opening_spot

    # Adjust moneyness for 'PE' type options
    df.loc[df['Type'] == 'PE', 'moneyness'] = 1 / df['moneyness']

    # Moneyness in percentage terms
    df['moneyness'] = (df['moneyness'] - 1) * 100

    # Spread in percentage
    df['spread_pct'] = df['bid_ask_spread'] / df['mid_price'] * 100

    # Define the bucketing function
    def bucket_moneyness(value):
        if value >= 0.5:
            return min(int(np.floor(value - 0.5) + 1), 11)  # Bucketing for positive values
        elif value <= -0.5:
            return max(int(np.ceil(value + 0.5) - 1), -11)  # Bucketing for negative values
        else:
            return 0  # Values between -0.5 and 0.5 are set to 0

    # Apply bucketing
    df['moneyness_bucket'] = df['moneyness'].apply(bucket_moneyness)

    # Function to calculate liquidity
    def calculate_liquidity(group):
        return (1 - (group['BidPrice'].diff() == 0).sum() / len(group)) * 100

    # Function to calculate spread metrics (including avg_spread_value)
    def calculate_spread_metrics(group):
        max_idx = group['spread_pct'].idxmax()
        min_idx = group['spread_pct'].idxmin()
        
        return pd.Series({
            'max_spread_value': group.loc[max_idx, 'spread_pct'],
            'max_spread_time': group.loc[max_idx, 'Date Time'].time(),
            'min_spread_value': group.loc[min_idx, 'spread_pct'],
            'min_spread_time': group.loc[min_idx, 'Date Time'].time(),
            'avg_spread_value': group['spread_pct'].mean()  # Average spread value
        })

    # Function to calculate price problem percentage
    def calculate_price_problem_pct(group):
        return (group['price_problem'] == True).sum() / len(group)

    # Group by 'moneyness_bucket' and 'Type' and compute metrics
    # Compute group metrics with include_groups=False to avoid the DeprecationWarning
    # liquidity = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_liquidity, include_groups=False).reset_index(name='liquidity')
    # spread_metrics = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_spread_metrics, include_groups=False).reset_index()
    # price_problem_pct = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_price_problem_pct, include_groups=False).reset_index(name='price_problem_pct')
    liquidity = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_liquidity).reset_index(name='liquidity')
    spread_metrics = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_spread_metrics).reset_index()
    price_problem_pct = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_price_problem_pct).reset_index(name='price_problem_pct')


    # Merge all computed values
    result = liquidity.merge(spread_metrics, on=['moneyness_bucket', 'Type']).merge(price_problem_pct, on=['moneyness_bucket', 'Type'])

    # Convert to dictionary with dynamic variable names
    result_dict = {'Date': file_date}  # Add the Date column

    for _, row in result.iterrows():
        key_prefix = f"{row['Type']}_{row['moneyness_bucket']}"
        
        result_dict[f"liq_{key_prefix}"] = row['liquidity']
        result_dict[f"spread_{key_prefix}_max_value"] = row["max_spread_value"]
        result_dict[f"spread_{key_prefix}_max_time"] = row["max_spread_time"]
        result_dict[f"spread_{key_prefix}_min_value"] = row["min_spread_value"]
        result_dict[f"spread_{key_prefix}_min_time"] = row["min_spread_time"]
        result_dict[f"spread_{key_prefix}_avg_value"] = row["avg_spread_value"]
        result_dict[f"price_problem_pct_{key_prefix}"] = row["price_problem_pct"]

    return result_dict



In [5]:
# all_results = []

# for file_path in csv_path_list:
#     daily_result = process_csv(file_path)
#     all_results.append(daily_result)

# # Convert list of dictionaries into a DataFrame
# final_df = pd.DataFrame(all_results)

# # Convert 'Date' column to datetime format (ensures correct sorting)
# final_df['Date'] = pd.to_datetime(final_df['Date'])

# # Sort DataFrame by 'Date' in ascending order (earliest to latest)
# final_df = final_df.sort_values(by='Date', ascending=True).reset_index(drop=True)

# # Save the result as a CSV (optional)
# final_df.to_csv("/home/hsph/Downloads/DataStore/SS/ASIANPAINT/2024/data_report.csv", index=False)

# # Display the final DataFrame
# final_df.head()

## Processing multiple Stocks in one go

In [10]:
folder_path_list = glob.glob("/home/cloudcraftz/Music/OneDrive_2_22-04-2025/*/*/")
folder_path_list

['/home/cloudcraftz/Music/OneDrive_2_22-04-2025/ICICIGI/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/HINDUNILVR/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/HDFCLIFE/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/INDUSINDBK/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/HDFCBANK/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/LALPATHLAB/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/HEROMOTOCO/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/INDHOTEL/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/ICICIPRULI/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/ICICIBANK/2024_new/',
 '/home/cloudcraftz/Music/OneDrive_2_22-04-2025/IDFCFIRSTB/2024_new/']

In [11]:
# folder_path_list = glob.glob("/home/hsph/Downloads/OneDrive_3_4-6-2025/*/2024/2024_new/")
folder_path_list = glob.glob("/home/cloudcraftz/Music/OneDrive_2_22-04-2025/*/*/")

for folder_path in folder_path_list:
    csv_path_list = glob.glob(folder_path + "*.csv")
    # print(csv_path_list)
    stock_name = folder_path.split('/')[-3]
    all_results = []

    for file_path in csv_path_list:
        daily_result = process_csv(file_path)
        all_results.append(daily_result)

    # Convert list of dictionaries into a DataFrame
    final_df = pd.DataFrame(all_results)

    # Convert 'Date' column to datetime format (ensures correct sorting)
    final_df['Date'] = pd.to_datetime(final_df['Date'])

    # Sort DataFrame by 'Date' in ascending order (earliest to latest)
    final_df = final_df.sort_values(by='Date', ascending=True).reset_index(drop=True)

    # Save the result as a CSV (optional)
    final_df.to_csv(f"/home/cloudcraftz/Downloads/data_reports/2024/{stock_name}.csv", index=False)

    print("Report Generated : ", stock_name)

Report Generated :  ICICIGI
Report Generated :  HINDUNILVR
Report Generated :  HDFCLIFE
Report Generated :  INDUSINDBK
Report Generated :  HDFCBANK
Report Generated :  LALPATHLAB
Report Generated :  HEROMOTOCO
Report Generated :  INDHOTEL
Report Generated :  ICICIPRULI
Report Generated :  ICICIBANK
Report Generated :  IDFCFIRSTB


In [10]:
import pandas as pd
from glob import glob
df = pd.read_csv("/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/notebooks/single_stocks_ststus_sumegh.csv")

In [12]:
df.shape

(94, 2)

In [14]:
list_of_PL = df['PRIORITY LIST'].to_list()
len(list_of_PL)

94

In [18]:
import glob

file_list = glob.glob("/home/cloudcraftz/Downloads/data_reports/2024/*.csv")
len(file_list)

89

In [24]:
reported_list = [file.split("/")[-1].split('.')[0] for file in file_list]
len(reported_list)

89

In [26]:
for i in reported_list:
    if i not in list_of_PL:
        print(i)

In [27]:
for i in list_of_PL:
    if i not in reported_list:
        print(i)

BOSCHLTD
ITC
LTF
M&MFIN
VOLTAS


In [37]:
df_sumegh = pd.read_csv("/home/cloudcraftz/Downloads/tiger_test/sumegh_test.txt", header=1)
df_sumegh['underlying'] = df_sumegh['20240101_BRITANNIA_HistoricData.txt'].apply(lambda x: x.split("_")[1])
ll_list = df_sumegh['underlying'].unique()
ll_list

array(['CANFINHOME', 'CHOLAFIN', 'CIPLA', 'COLPAL', 'CROMPTON',
       'BOSCHLTD', 'BRITANNIA', 'BANKNIFTY', 'NIFTY'], dtype=object)

In [38]:
for i in ll_list:
    if i not in reported_list:
        print(i)

BOSCHLTD
BANKNIFTY
NIFTY


In [39]:
df_sumegh = pd.read_csv("/home/cloudcraftz/sumegh.txt", header=1)
def extract_underlying(value):
    try:
        return value.split("_")[1]
    except Exception as e:
        print(f"Error processing value: {value} -> {e}")
        return None  # or any fallback value

df_sumegh['underlying'] = df_sumegh['20180102_BANKNIFTY_HistoricData.txt'].apply(extract_underlying)

Error processing value: BIOCON.tar -> list index out of range
Error processing value: BOSCHLTD.tar -> list index out of range
Error processing value: Errors -> list index out of range
Error processing value: oem@192.168.0.165 -> list index out of range
Error processing value: sumegh.txt -> list index out of range


In [40]:
ll_list2 = df_sumegh['underlying'].unique()
ll_list2

array(['BANKNIFTY', 'NIFTY', 'ABB', 'AUBANK', 'SBIN', 'AARTIIND',
       'ABBOTINDIA', 'ABCAPITAL', 'ABFRL', 'ACC', 'ADANIENSOL',
       'ADANIENT', 'ADANIGREEN', 'ADANIPORTS', 'ALKEM', 'AMBUJACEM',
       'ANGELONE', 'APLAPOLLO', 'APOLLOHOSP', 'APOLLOTYRE', 'ASHOKLEY',
       'ASIANPAINT', 'ASTRAL', 'ATGL', 'ATUL', 'AUROPHARMA', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BALKRISIND',
       'BANDHANBNK', 'BANKBARODA', 'BANKINDIA', 'BATAINDIA', 'BEL',
       'BERGEPAINT', 'BHARATFORG', 'BHARTIARTL', 'BHEL', 'BIOCON',
       'BOSCHLTD', 'BRITANNIA', 'BSE', 'BSOFT', 'CANFINHOME', 'CESC',
       'CGPOWER', 'CHOLAFIN', 'CIPLA', 'COLPAL', 'CROMPTON', 'CUMMINSIND',
       'DABUR', 'DIVISLAB', 'DMART', 'EICHERMOT', 'ESCORTS', 'FEDERALBNK',
       'GODREJCP', 'GUJGASLTD', 'HAVELLS', 'HCLTECH', 'HDFCAMC',
       'HDFCBANK', 'HDFCLIFE', 'HEROMOTOCO', 'HINDUNILVR', 'ICICIBANK',
       'ICICIGI', 'ICICIPRULI', 'IDFCFIRSTB', 'INDHOTEL', 'INDIGO',
       'INDUSINDBK', 'INDUSTOWER'

In [45]:
Download_list = []
for i in ll_list2:
    if i not in reported_list:
        Download_list.append(i)

In [ ]:
['AARTIIND', 'ABFRL', 'ACC', 'ADANIENSOL', 'ADANIENT', 'ADANIGREEN', 'ADANIPORTS', 'ALKEM', 'AMBUJACEM', 'ANGELONE', 'APLAPOLLO', 'APOLLOTYRE', 'ATGL', 'ATUL', 'AUROPHARMA', 'BAJAJFINSV', 'BANDHANBNK', 'BANKBARODA', 'BANKINDIA', 'BATAINDIA', 'BEL', 'BHARATFORG', 'BHEL', 'BSE', 'BSOFT', 'CESC', 'CGPOWER', 'DMART', 'HIST', 'ITC']

In [ ]:
['AARTIIND' 'ABFRL' 'ACC' 'ADANIENSOL' 'ADANIENT' 'ADANIGREEN' 'ADANIPORTS' 'ALKEM' 'AMBUJACEM' 'ANGELONE' 'APLAPOLLO' 'APOLLOTYRE' 'ATGL' 'ATUL' 'AUROPHARMA' 'BAJAJFINSV' 'BANDHANBNK' 'BANKBARODA' 'BANKINDIA' 'BATAINDIA' 'BEL' 'BHARATFORG' 'BHEL' 'BSE' 'BSOFT' 'CESC' 'CGPOWER' 'DMART' 'HIST']

In [44]:
# Download_list

In [ ]:
df = pd.read_csv("/home/cloudcraftz/Downloads/SingleStockDerivatives(Sheet1).csv")
df[(df['STATUS']=='Completed')&(df['IN PRIORITY LIST']==True)][['SYMBOL']]

In [48]:
ll = df[(df['STATUS']=='Completed')][['SYMBOL']]

In [50]:
not_ll = df[(df['STATUS']!='Completed')][['SYMBOL']]

In [51]:
not_ll

,SYMBOL
31,BANDHANBNK
38,BHEL
39,BPCL
44,CESC
45,CGPOWER
...,...
221,IDEA
223,WIPRO
224,YESBANK
225,ZOMATO


In [55]:
df_sumegh1 = pd.read_csv("/home/cloudcraftz/sumegh_test1.txt", header=1)
def extract_underlying(value):
    try:
        return value.split("_")[1]
    except Exception as e:
        print(f"Error processing value: {value} -> {e}")
        return None  # or any fallback value

df_sumegh1['underlying'] = df_sumegh1['20240101_GUJGASLTD_HistoricData.txt'].apply(extract_underlying)

Error processing value: Errors -> list index out of range


In [58]:
ll_list3 = df_sumegh1['underlying'].unique()
ll_list3

array(['HCLTECH', 'HDFCAMC', 'GODREJCP', 'GUJGASLTD', 'NIFTY', None,
       'test1.txt'], dtype=object)

# Tasks
### Liquidity using BID-ASK movement (median threshold)
### Using Volume based approach - Last Traded Volume and Last traded Price
### Price Filter : Step by Step : See if the LTP between Bid and Ask : If volume is not zero (ie. LTP is also non zero)
### DDG Approach : See the stream of BID/ASK prices : If they are sustained for some time, then they are reasonable, otherwise they are not

In [1]:
import yfinance as yf

# Download data for BAJFINANCE from Jan 2023 to Dec 2024
data = yf.download("BAJFINANCE.NS", start="2023-01-01", end="2025-01-01")  # include end date buffer

# Save to CSV
data.to_csv("BAJFINANCE_Jan2023_Dec2024.csv")


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BAJFINANCE.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


In [2]:
import pandas as pd

In [15]:
df = pd.read_csv("/home/cloudcraftz/Downloads/Untitled spreadsheet - Sheet1_bajaj.csv")

In [16]:
df['Date'] = pd.to_datetime(df['Date'])

In [17]:
df['Date'] = df['Date'].dt.date

In [18]:
df

,Date,Open,High,Low,Close,Volume
0,2023-02-01,3617.00,3620.00,3520.45,3573.95,471328
1,2023-03-01,3562.50,3620.00,3562.50,3601.70,218996
2,2023-04-01,3588.00,3602.55,3543.50,3552.85,250071
3,2023-05-01,3568.00,3632.50,3558.35,3621.15,294998
4,2023-06-01,3618.00,3648.00,3598.00,3642.25,173591
...,...,...,...,...,...,...
485,2024-12-24,8745.00,8854.00,8731.00,8778.05,297827
486,2024-12-26,8810.00,8898.65,8797.60,8878.50,240675
487,2024-12-27,8896.25,9179.15,8896.25,8928.30,713799
488,2024-12-30,8928.00,8962.45,8737.55,8779.90,1065253


In [19]:
df.to_csv("/home/cloudcraftz/Downloads/BAJAJAUTO_2023_2024_EOD.csv", index=False)